# NB05a Quantification

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB05a_Quantification.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB05a/NB05a_Quantification.ipynb)

1. Quantification Theory
    1. Targeted quantification
    2. (i) Targeted acquisition at peptide
    3. (ii) Targeted data analysis at peptide ion level
2. References

## Quantification Theory

To estimate the amount of individual proteins in complex mixtures, all peptide signals corresponding to a common protein serve as a 
proxy for their abundance. Peptide information needs to be obtained from multidimensional signal data detected by the mass spectrometer. 
All signals generated from one peptide ion species, often referred to as peptide feature, need to be grouped to form a three-dimensional peak 
along the m/z, ion intensity, and retention time dimension. This process is generally defined as peak detection or feature detection. 
Peak detection algorithms are a set of rules defining how neighboring signal points are joined. Whether noise filtering is done before or after 
peak detection strongly depends on the peak detection algorithm. Traditional approaches mainly focused on signal amplitude neglecting 
characteristic peak shapes as a common feature of chromatographic or spectroscopic peaks. These algorithms are prone to miss detection of low 
intensity peaks with a signal strength close to the noise level. To overcome these issues, techniques like smoothing, shape-matching and curve 
fitting are often implemented and applied. At the time, the most promising approach to do shape-matching and noise reduction in one step uses the 
continuous wavelet transformation (CWT).

In general, a CWT based approach describes a family of time-frequency-transformations often used in data compression and feature detection. 
The term is coined by the use of a wavelet, as a basis function which is “compared” to the signal. The point of highest correlation between the 
basis function and the signal reflects the location of the peak present. Due to the fact that MS derived peaks often follow the shape of a 
gaussian distribution, the *Mexican Hat* wavelet as the negative normalized second derivative of the Gaussian distribution is perfectly 
suited to find the peptide feature.

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/Wavelets.png)

**Figure 5: Schematic representation of the ‘Haar’-wavelet (blue) and the ‘Mexican Hat’- wavelet (green). **
The ‘Haar’-wavelet is named after its discoverer Alfred Haar and represents the first wavelet ever to be described. The ‘Mexican Hat’- or ‘Ricker’-wavelet is 
frequently used in the fields of signal detection and compression.

Depending on the quantification approach, the peptide features used for protein quantification might differ. In case of isotopic labeling, 
quantification means pairing features with the proper mass shift according to the utilized label. It is essential to account for the frequency 
of label incorporation when calculating the mass shift for the utilized label. Taking the ICAT method as an example, by which a heavy/light 
difference of 9 Dalton per cysteine is incorporated, the total mass shift is 9 Dalton times the number of cysteine within the sequence. 
Consequently, pairing peptide features for 15N labeling is even more challenging, as the mass shift is less discrete. Using stable 
isotope labeling, different peptide feature pairs belonging to the same protein can be treated as technical replicates and averaged to gain 
protein quantification. In contrast, the sum of all extracted peptide signals results in a label-free protein quantiﬁcation. Spectral counting 
computes abundance values from the number of times a peptide was successfully identiﬁed by tandem mass spectrometry (MS/MS) and combines all 
these events per protein. The spectral counting values can be normalized by the number of peptides theoretically expected from the particular 
protein. 

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/ComputationalProteinQuantification.png)

**Figure 6: Computational strategy of peptide and protein quantiﬁcation on based on stable isotope labeling or by label-free quantiﬁcation.**
(A) Label-free methods compare corresponding peptide abundances over different MS runs. The abundance is either 
estimated by the elution proﬁle les of the pep de ions or (B) in case of spectral counting, by the number of times a peptide was 
successfully identiﬁed (MS2). In contrast, methods based on differential stable isotope labeling analyze peptides pairs detected by 
their characteristic mass diﬀerence Δm/z. The abundance is estimated by the ratio of their corresponding elution proﬁles (C). Isobaric 
tagging methods (D) compare the reporter ion abundances in the fragmentation spectrum.
### Targeted quantification

Targeted proteomics has gained significant popularity in mass spectrometry‐based protein quantification as a method to detect proteins of 
interest with high sensitivity, quantitative accuracy and reproducibility. The two major strategies of (i) targeted acquisition at peptide, 
and (ii) targeted data analysis at peptide ion level need to be distinguished.
###(i) Targeted acquisition at peptide

In multiple reaction monitoring (MRM or SRM for single/selected reaction monitoring) simply predefined transitions are recorded. 
Knowledge about the targeted transitions from precursor to their corresponding fragment ions are needed and predefined in the mass 
spectrometer. MRM can be performed rapidly and is highly specific even for low abundant peptide ions in complex mixtures, but with the 
drawback of a necessary bias in the sense that only predefined peptides are measured.
### (ii) Targeted data analysis at peptide ion level

Data‐independent acquisition at the peptide level makes it possible to acquire peptide data for virtually all peptide ions present in a sample. 
In this strategy, a high‐resolution mass analyzer—such as an orbitrap or a time‐of‐flight—is used to constantly sample the full mass range 
at the peptide level during the entire chromatographic gradient. In a subsequent step, precursor ion chromatograms can be extracted by targeted 
data analysis. Those extracted-ion chromatogram (XIC) can be obtained to calculate the area under the curve and used for peptide quantification.

Let’s start and extract a XIC…


In [1]:
#r "nuget: FSharp.Stats, 0.4.3"
#r "nuget: BioFSharp, 2.0.0-beta5"
#r "nuget: BioFSharp.IO, 2.0.0-beta5"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: System.Data.SQLite, 1.0.113.7"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: MzIO, 0.1.1"
#r "nuget: MzIO.SQL, 0.1.4"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: Deedle, 3.0.0"
#r "nuget: Deedle.Interactive, 3.0.0"

open Plotly.NET
open FSharp.Stats
open BioFSharp
open System.IO
open System.Data.SQLite
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL
open Deedle
open Deedle.Interactive


Installed Packages BioFSharp, 2.0.0-beta5 BioFSharp.IO, 2.0.0-beta5 BioFSharp.Mz, 0.1.5-beta Deedle, 3.0.0 Deedle.Interactive, 3.0.0 FSharp.Stats, 0.4.3 MzIO, 0.1.1 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.4 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0 System.Data.SQLite, 1.0.113.7

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

Loading extensions from `/home/paulinehans/.nuget/packages/deedle.interactive/3.0.0/lib/netstandard2.1/Deedle.Interactive.dll`

In the last Notebook we get at the End our Scoring.tsv file. The first step is now, to load this file and get accsess to it.

In [2]:
// Code-Block 1id

let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"/home/paulinehans/Dokumente/scoring.tsv"|]

let data = Deedle.Frame.ReadCsv (path, hasHeaders =  true,  separators = "\t")
data

0,->,2,832.5058404091601,7.463680414533675,LLFEALK,19.931816666667,sample=1 period=1 cycle=2174 experiment=2
1,->,2,1197.6757592657698,9.395028227159235,LVFPEEVLPR,19.84665,sample=1 period=1 cycle=2160 experiment=2
2,->,2,832.5058404091601,7.302567421626155,LLFEALK,19.82215,sample=1 period=1 cycle=2158 experiment=2
3,->,2,1197.6757592657698,9.785705961420422,LVFPEEVLPR,19.737,sample=1 period=1 cycle=2144 experiment=2
4,->,2,832.5058404091601,6.407189581270389,LLFEALK,19.7175,sample=1 period=1 cycle=2143 experiment=2
:,,...,...,...,...,...,...
1208,->,2,832.5058404091601,6.994930950008397,LLFEALK,6.964933333333,sample=1 period=1 cycle=1365 experiment=2
1209,->,2,832.5058404091601,6.568610099543873,LLFEALK,6.864933333333,sample=1 period=1 cycle=1351 experiment=2
1210,->,2,832.5058404091601,7.796326175038813,LLFEALK,6.764283333333,sample=1 period=1 cycle=1334 experiment=2
1211,->,2,832.5058404091601,6.685626444937399,LLFEALK,6.654283333333,sample=1 period=1 cycle=1318 experiment=2
1212,->,2,832.5058404091601,6.931277680938116,LLFEALK,6.5488,sample=1 period=1 cycle=1300 experiment=2


We have now a Dataframe, looks like a table, works like a table. What we do now is relative Quantification in the form of XICs. For that we need several types of information: 
- precursor mass
- charge 
- peptide sequence
- retention time
- labeled data (15N)

(and the retention time index but later more to that). The Information is stored in the Dataframe, so the goal is to extract the data out of it so that we can work with it. 

In [3]:
let extract = 
    data 
    |> Frame.sliceCols([|"Mass"; "Charge"; "PeptideSequence"; "RetentionTime"; "MatchingScore"|])
extract

0,->,832.5058404091601,2,LLFEALK,19.931816666667,7.463680414533675
1,->,1197.6757592657698,2,LVFPEEVLPR,19.84665,9.395028227159235
2,->,832.5058404091601,2,LLFEALK,19.82215,7.302567421626155
3,->,1197.6757592657698,2,LVFPEEVLPR,19.737,9.785705961420422
4,->,832.5058404091601,2,LLFEALK,19.7175,6.407189581270389
:,,...,...,...,...,...
1208,->,832.5058404091601,2,LLFEALK,6.964933333333,6.994930950008397
1209,->,832.5058404091601,2,LLFEALK,6.864933333333,6.568610099543873
1210,->,832.5058404091601,2,LLFEALK,6.764283333333,7.796326175038813
1211,->,832.5058404091601,2,LLFEALK,6.654283333333,6.685626444937399
1212,->,832.5058404091601,2,LLFEALK,6.5488,6.931277680938116


Nice! We have now our columns of interest. Now we will get the values out of the columns

In [4]:

let precursorMass = extract.GetColumn<float>("Mass").Values |> Seq.toArray
let rt   = extract.GetColumn<float>("RetentionTime").Values |> Seq.toArray
let charge = extract.GetColumn<int>("Charge").Values |> Seq.toArray
let sequence = extract.GetColumn<string>("PeptideSequence").Values |> Seq.toArray
let score = extract.GetColumn<float>("MatchingScore").Values |> Seq.toArray


The retention-time index is derived from MS1 spectra and is essential because our XIC is built from MS2 information; to align MS2 data with the corresponding MS1 retention-time values, we need this retention-time index. Logically we take our .mzlite file from the last notebook and extract the MS1RTI.

In [5]:
let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"downloads/qPgr1_1_5 1.mzlite"|]

let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)
let cn = mzReader.Open()
let transaction = mzReader.BeginTransaction()

// Indexes all spectra of the related sample run
let idx = MzIO.Processing.Query.getMS1RTIdx mzReader runID
idx 

let rtIndex = idx |> Seq.toArray |> Array.rev
rtIndex

index value 0 [rt=19.997416666667, spectrumID='sample=1 period=1 cycle=2187 experiment=1'] Rt 19.997416666667 SpectrumID sample=1 period=1 cycle=2187 experiment=1 1 [rt=19.992416666667, spectrumID='sample=1 period=1 cycle=2186 experiment=1'] Rt 19.992416666667 SpectrumID sample=1 period=1 cycle=2186 experiment=1 2 [rt=19.987416666667, spectrumID='sample=1 period=1 cycle=2185 experiment=1'] Rt 19.987416666667 SpectrumID sample=1 period=1 cycle=2185 experiment=1 3 [rt=19.982316666667, spectrumID='sample=1 period=1 cycle=2184 experiment=1'] Rt 19.982316666667 SpectrumID sample=1 period=1 cycle=2184 experiment=1 4 [rt=19.977316666667, spectrumID='sample=1 period=1 cycle=2183 experiment=1'] Rt 19.977316666667 SpectrumID sample=1 period=1 cycle=2183 experiment=1 5 [rt=19.9723, spectrumID='sample=1 period=1 cycle=2182 experiment=1'] Rt 19.9723 SpectrumID sample=1 period=1 cycle=2182 experiment=1 6 [rt=19.9672, spectrumID='sample=1 period=1 cycle=2181 experiment=1'] Rt 19.9672 SpectrumID sample=1 period=1 cycle=2181 experiment=1 7 [rt=19.9622, spectrumID='sample=1 period=1 cycle=2180 experiment=1'] Rt 19.9622 SpectrumID sample=1 period=1 cycle=2180 experiment=1 8 [rt=19.9572, spectrumID='sample=1 period=1 cycle=2179 experiment=1'] Rt 19.9572 SpectrumID sample=1 period=1 cycle=2179 experiment=1 9 [rt=19.952083333333, spectrumID='sample=1 period=1 cycle=2178 experiment=1'] Rt 19.952083333333 SpectrumID sample=1 period=1 cycle=2178 experiment=1 10 [rt=19.947033333333, spectrumID='sample=1 period=1 cycle=2177 experiment=1'] Rt 19.947033333333 SpectrumID sample=1 period=1 cycle=2177 experiment=1 11 [rt=19.942033333333, spectrumID='sample=1 period=1 cycle=2176 experiment=1'] Rt 19.942033333333 SpectrumID sample=1 period=1 cycle=2176 experiment=1 12 [rt=19.93695, spectrumID='sample=1 period=1 cycle=2175 experiment=1'] Rt 19.93695 SpectrumID sample=1 period=1 cycle=2175 experiment=1 13 [rt=19.91745, spectrumID='sample=1 period=1 cycle=2174 experiment=1'] Rt 19.91745 SpectrumID sample=1 period=1 cycle=2174 experiment=1 14 [rt=19.91235, spectrumID='sample=1 period=1 cycle=2173 experiment=1'] Rt 19.91235 SpectrumID sample=1 period=1 cycle=2173 experiment=1 15 [rt=19.907333333333, spectrumID='sample=1 period=1 cycle=2172 experiment=1'] Rt 19.907333333333 SpectrumID sample=1 period=1 cycle=2172 experiment=1 16 [rt=19.902333333333, spectrumID='sample=1 period=1 cycle=2171 experiment=1'] Rt 19.902333333333 SpectrumID sample=1 period=1 cycle=2171 experiment=1 17 [rt=19.897233333333, spectrumID='sample=1 period=1 cycle=2170 experiment=1'] Rt 19.897233333333 SpectrumID sample=1 period=1 cycle=2170 experiment=1 18 [rt=19.892216666667, spectrumID='sample=1 period=1 cycle=2169 experiment=1'] Rt 19.892216666667 SpectrumID sample=1 period=1 cycle=2169 experiment=1 19 [rt=19.887216666667, spectrumID='sample=1 period=1 cycle=2168 experiment=1'] Rt 19.887216666667 SpectrumID sample=1 period=1 cycle=2168 experiment=1 (2167 more)

We're almost done with the data we need. The last puzzle piece is 15N labled data, we're creating this exactly like in the "Isotopic Distribution Notebook"

In [6]:
// Converts a peptide BioSeq into a molecular formula used for mass calculations.
let toFormula bioseq =  
    bioseq
    |> BioSeq.toFormula
    // peptides are hydrolysed in the mass spectrometer, so we add H2O
    |> Formula.add Formula.Table.H2O

// Calculates the monoisotopic mass of a fully 15N-labeled peptide sequence.
let label formula =
    let forumlaBioItem = 
        formula
        // Convert amino-acid string (e.g. "PEPTIDE") to BioSeq.
        |> BioSeq.ofAminoAcidString
        // Build the peptide formula (including added H2O).
        |> toFormula
    // Replace all natural nitrogen (14N) with heavy nitrogen (15N).
    Formula.replaceElement
        forumlaBioItem
        Elements.Table.N
        Elements.Table.Heavy.N15
    // Return monoisotopic mass of the labeled formula.
    |> Formula.monoisoMass

### Simply lovely 
Now we can start with the realtive Quantification! 

The first thing we do is group our information, because every peptide with the same charge and sequence will grouped because the same sequence can be “hit” multiple times in a single run. After that we take the best score (remember here the last task in Peptide Identification Notebook) within a group. For a better handling we create a record type, which stores all our Information (like a box that you can carry around). 

In [7]:
// Record type with all the needed Information
type ConsensLookup = {
    PeptideSequence: string
    RetentionTime: float
    Charge: int
    LabeledMass: float
    PrecursorMass: float
}

// Build one row per index by combining values from all parallel arrays.
let rows = [|
        for i = 0 to rt.Length - 1 do
            // Read sequence and charge at the current index.
            let s = sequence.[i]
            let c = charge.[i]
            // Compute theoretical labeled m/z from sequence and charge.
            let lM = BioFSharp.Mass.toMZ (s |> label) c
            // Store all relevant values as one tuple.
            s, rt.[i], c, lM, precursorMass.[i], score.[i]
    |]
// Build a consensus lookup: one best entry per (sequence, charge) pair.
let consensLookup =
    rows
    // Group rows by peptide sequence and charge state.
    |> Array.groupBy (fun (sequence,rt,charge,labeledTheoMass,precMass,score) -> sequence, charge)
    |> Array.map (fun ((sequence,charge), grouped) ->
        // Keep the row with the highest score in each group.
        grouped
        |> Array.maxBy (fun (sequence,rt,charge,labeledTheoMass,precMass,score) -> score)
        // Convert the selected tuple into a typed record.
        |> (fun (sequence,rt,charge,labeledTheoMass,precMass,score) -> 
            {
                PeptideSequence = sequence
                RetentionTime = rt
                Charge = charge
                LabeledMass = labeledTheoMass
                PrecursorMass = precMass
            }
        )
        //Backup (pls ignore)
        // let medianLabeledTheoMass =
        //     grouped
        //     |> Array.map (fun (sequence,rt,charge,labeledTheoMass,precMass) -> labeledTheoMass)
        //     |> Array.median
        // let medianRT =
        //     grouped
        //     |> Array.map (fun (sequence,rt,charge,labeledTheoMass,precMass) -> rt)
        //     |> Array.median
        // let medianPrecMass =
        //     grouped
        //     |> Array.map (fun (sequence,rt,charge,labeledTheoMass,precMass) -> precMass)
        //     |> Array.median

        // {
        //     PeptideSequence = sequence
        //     RetentionTime = medianRT
        //     Charge = charge
        //     LabeledMass = medianLabeledTheoMass
        //     PrecursorMass = medianPrecMass
        // }
    )
consensLookup

index value 0 { PeptideSequence = "LLFEALK"\n RetentionTime = 6.764283333\n Charge = 2\n LabeledMass = 421.2483362\n PrecursorMass = 832.5058404 } PeptideSequence LLFEALK RetentionTime 6.764283333333 Charge 2 LabeledMass 421.24833624499 PrecursorMass 832.5058404091601 1 { PeptideSequence = "LVFPEEVLPR"\n RetentionTime = 9.035983333\n Charge = 2\n LabeledMass = 606.3258829\n PrecursorMass = 1197.675759 } PeptideSequence LVFPEEVLPR RetentionTime 9.035983333333 Charge 2 LabeledMass 606.3258829067951 PrecursorMass 1197.6757592657698 2 { PeptideSequence = "AVSLVLPSLK"\n RetentionTime = 19.06486667\n Charge = 2\n LabeledMass = 519.3152093\n PrecursorMass = 1025.648482 } PeptideSequence AVSLVLPSLK RetentionTime 19.064866666667 Charge 2 LabeledMass 519.315209325455 PrecursorMass 1025.64848188989 3 { PeptideSequence = "VPLILGIWGGK"\n RetentionTime = 9.094633333\n Charge = 2\n LabeledMass = 583.341336\n PrecursorMass = 1151.706665 } PeptideSequence VPLILGIWGGK RetentionTime 9.094633333333 Charge 2 LabeledMass 583.341336010365 PrecursorMass 1151.70666547291 4 { PeptideSequence = "DTDILAAFR"\n RetentionTime = 18.09343333\n Charge = 2\n LabeledMass = 517.2514907\n PrecursorMass = 1020.52401 } PeptideSequence DTDILAAFR RetentionTime 18.093433333333 Charge 2 LabeledMass 517.25149065303 PrecursorMass 1020.52400965164 5 { PeptideSequence = "YWTMWK"\n RetentionTime = 18.78905\n Charge = 2\n LabeledMass = 462.2017562\n PrecursorMass = 913.4156455 } PeptideSequence YWTMWK RetentionTime 18.78905 Charge 2 LabeledMass 462.2017562402751 PrecursorMass 913.41564550633 6 { PeptideSequence = "LSIFETGIK"\n RetentionTime = 18.82688333\n Charge = 2\n LabeledMass = 509.2773995\n PrecursorMass = 1006.569897 } PeptideSequence LSIFETGIK RetentionTime 18.826883333333 Charge 2 LabeledMass 509.2773995415401 PrecursorMass 1006.56989721546 7 { PeptideSequence = "NLALELVR"\n RetentionTime = 17.40263333\n Charge = 2\n LabeledMass = 470.2669438\n PrecursorMass = 926.5549159 } PeptideSequence NLALELVR RetentionTime 17.402633333333 Charge 2 LabeledMass 470.2669437566001 PrecursorMass 926.55491585878 8 { PeptideSequence = "TWFDDADDWLR"\n RetentionTime = 10.8692\n Charge = 2\n LabeledMass = 728.2912275\n PrecursorMass = 1438.615344 } PeptideSequence TWFDDADDWLR RetentionTime 10.8692 Charge 2 LabeledMass 728.29122753092 PrecursorMass 1438.61534383382 9 { PeptideSequence = "VPLFIGSK"\n RetentionTime = 13.09923333\n Charge = 2\n LabeledMass = 435.2523032\n PrecursorMass = 859.5167394 } PeptideSequence VPLFIGSK RetentionTime 13.099233333333 Charge 2 LabeledMass 435.252303210125 PrecursorMass 859.51673944603 10 { PeptideSequence = "NILLNEGIR"\n RetentionTime = 12.74891667\n Charge = 2\n LabeledMass = 528.2854424\n PrecursorMass = 1040.597843 } PeptideSequence NILLNEGIR RetentionTime 12.748916666667 Charge 2 LabeledMass 528.28544237001 PrecursorMass 1040.5978432988 11 { PeptideSequence = "FLAIDAINK"\n RetentionTime = 13.1604\n Charge = 2\n LabeledMass = 508.2760842\n PrecursorMass = 1003.570232 } PeptideSequence FLAIDAINK RetentionTime 13.1604 Charge 2 LabeledMass 508.27608416510503 PrecursorMass 1003.57023156919 12 { PeptideSequence = "VAELLDFK"\n RetentionTime = 18.47523333\n Charge = 2\n LabeledMass = 472.2525002\n PrecursorMass = 933.5171334 } PeptideSequence VAELLDFK RetentionTime 18.475233333333 Charge 2 LabeledMass 472.25250017069504 PrecursorMass 933.51713336717 13 { PeptideSequence = "ALQNTVLK"\n RetentionTime = 10.35406667\n Charge = 2\n LabeledMass = 449.2551518\n PrecursorMass = 885.5283668 } PeptideSequence ALQNTVLK RetentionTime 10.354066666667 Charge 2 LabeledMass 449.25515175939506 PrecursorMass 885.5283667577701 14 { PeptideSequence = "IYLDISDDIK"\n RetentionTime = 13.53471667\n Charge = 2\n LabeledMass = 603.2999532\n PrecursorMass = 1193.61797 } PeptideSequence IYLDISDDIK RetentionTime 13.534716666667 Charge 2 LabeledMass 603.299953182455 PrecursorMass 1193.61796960389 15 { PeptideSequence = "LSELLGKPVTK"\n RetentionTime = 11.298\n Charge = 2\n LabeledMass = 5

The output of the function consensLookup is crystal clear: You have PeptideSequence, rt, charge, labledTheoMass and precursorMass. The next step is that we create a query to the database to extract the intensities of all peaks that are +/-5 min of our retention time and within 0.04 m/z of our peptide of interest. After we are done, we close the connection to the database. Dont confuse yourself with the helper function at the top, this is for a so-called baseline correction that is used for noise reduction. 

In [8]:
// Helper function to subtract an estimated baseline from intensity values.
// For very long traces (>500 points), skip correction and return original data.
let substractBaseLine (yData: float []) =
    if yData.Length > 500 then
        // No baseline correction for long signals (performance/robustness choice).
        yData
    else
        // Estimate baseline using asymmetric least squares (ALS).
        let baseLine =
            FSharp.Stats.Signal.Baseline.baselineAls 10 6 0.05 yData
            |> Array.ofSeq

        // Subtract baseline point-by-point and clamp negative values to zero.
        Array.map2 (fun y b ->
            let corrected = y - b
            if corrected < 0. then 0. else corrected
        ) yData baseLine

// the important part for you starts here
// Create retention-time query windows (center = consensus RT, tolerance = +/- 5).
let rtQuery = 
    consensLookup
    |> Array.map (fun lookup -> 
        MzIO.Processing.Query.createRangeQuery lookup.RetentionTime 5.)
// Create m/z query windows (center = labeled mass, tolerance = +/- 0.04).
let mzQuery = 
    consensLookup 
    |> Array.map (fun lookup -> 
        MzIO.Processing.Query.createRangeQuery lookup.LabeledMass 0.04
    )
// Pair each m/z query with its matching RT query (same peptide index).
let combine =  mzQuery  |> Array.zip rtQuery 
combine

index value 0 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 6.764283333333 LowValue 1.7642833333330001 HighValue 11.764283333333001 Item2 MzIO.Processing.RangeQuery LockValue 421.24833624499 LowValue 421.20833624499 HighValue 421.28833624499003 1 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 9.035983333333 LowValue 4.035983333333 HighValue 14.035983333333 Item2 MzIO.Processing.RangeQuery LockValue 606.3258829067951 LowValue 606.2858829067951 HighValue 606.365882906795 2 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 19.064866666667 LowValue 14.064866666667001 HighValue 24.064866666667 Item2 MzIO.Processing.RangeQuery LockValue 519.315209325455 LowValue 519.2752093254551 HighValue 519.355209325455 3 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 9.094633333333 LowValue 4.094633333333 HighValue 14.094633333333 Item2 MzIO.Processing.RangeQuery LockValue 583.341336010365 LowValue 583.301336010365 HighValue 583.381336010365 4 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 18.093433333333 LowValue 13.093433333333 HighValue 23.093433333333 Item2 MzIO.Processing.RangeQuery LockValue 517.25149065303 LowValue 517.2114906530301 HighValue 517.29149065303 5 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 18.78905 LowValue 13.78905 HighValue 23.78905 Item2 MzIO.Processing.RangeQuery LockValue 462.2017562402751 LowValue 462.16175624027505 HighValue 462.2417562402751 6 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 18.826883333333 LowValue 13.826883333333 HighValue 23.826883333333 Item2 MzIO.Processing.RangeQuery LockValue 509.2773995415401 LowValue 509.23739954154007 HighValue 509.3173995415401 7 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 17.402633333333 LowValue 12.402633333333 HighValue 22.402633333333 Item2 MzIO.Processing.RangeQuery LockValue 470.2669437566001 LowValue 470.22694375660006 HighValue 470.3069437566001 8 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 10.8692 LowValue 5.869199999999999 HighValue 15.8692 Item2 MzIO.Processing.RangeQuery LockValue 728.29122753092 LowValue 728.25122753092 HighValue 728.3312275309199 9 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 13.099233333333 LowValue 8.099233333333 HighValue 18.099233333333 Item2 MzIO.Processing.RangeQuery LockValue 435.252303210125 LowValue 435.21230321012496 HighValue 435.292303210125 10 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 12.748916666667 LowValue 7.748916666667 HighValue 17.748916666667 Item2 MzIO.Processing.RangeQuery LockValue 528.28544237001 LowValue 528.24544237001 HighValue 528.3254423700099 11 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 13.1604 LowValue 8.1604 HighValue 18.1604 Item2 MzIO.Processing.RangeQuery LockValue 508.27608416510503 LowValue 508.236084165105 HighValue 508.31608416510505 12 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 18.475233333333 LowValue 13.475233333333001 HighValue 23.475233333333 Item2 MzIO.Processing.RangeQuery LockValue 472.25250017069504 LowValue 472.212500170695 HighValue 472.29250017069506 13 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1 MzIO.Processing.RangeQuery LockValue 10.354066666667 LowValue 5.354066666667 HighValue 15.354066666667 Item2 MzIO.Processing.RangeQuery LockValue 449.25515175939506 LowValue 449.21515175939504 HighValue 449.2951517593951 14 (MzIO.Processing.RangeQuery, MzIO.Processing.RangeQuery) Item1

The output here gives you a tuple of rt and mass query, which is the input for the function which creates the XICs. Dont worry if this takes some minutes.

In [ ]:
// Build one XIC for each (retention time, m/z) pair from `combine`
let xics = 
    combine 
    |> Array.map (fun (rt, mz) -> 
        MzIO.Processing.Query.getXIC mzReader idx rt mz)
// Baseline-correct each XIC trace
let baselineCorrection = 
    xics
    |> Array.map (fun xic ->
        let intensity,mz,rt =
        // Split the XIC points into separate arrays
            xic |> Array.map (fun x -> x.Intensity),
            xic |> Array.map (fun x -> x.Mz),
            xic |> Array.map (fun x -> x.Rt)
        // Pair RT with baseline-corrected intensity
        Array.zip rt (substractBaseLine intensity)
    )
 
transaction.Dispose()

// Keep corrected traces together with consensus lookup data
let CorrAndRecord =  consensLookup |> Array.zip baselineCorrection

// Create one point chart per corrected trace
let xicChart =
    CorrAndRecord 
    |> Array.map (fun x -> 
        fst x
        |> Chart.Point 
        |> Chart.withXAxisStyle "Retention Time"
        |> Chart.withYAxisStyle "Intensity/Score"
        |> Chart.withSize (900.,900.)
    )
xicChart

### Hip Hip Hoorey!
We have our XICs (slay) Now we use the second derivative to identify peaks with our trace

In [32]:
// Detect peaks on each corrected XIC and keep its associated record info
let peaks = 
    CorrAndRecord
    |> Array.map (fun peak ->
        // peak is expected as: (correctedTrace, recordInfo)
        fst peak
        // correctedTrace is (rt, intensity)[] -> split into separate arrays
        |> Array.unzip
        |> (fun (ret, intensity) ->
            // Second-derivative peak detection on RT/intensity signal
            // args: threshold, minDistance, smoothingWindow
            FSharp.Stats.Signal.PeakDetection.SecondDerivative.getPeaks 0.1 2 13 ret intensity
        ), snd peak
    )
peaks |> Array.head

(FSharp.Stats.Signal.PeakDetection+IdentifiedPeak[], { PeptideSequence = "LLFEALK"\n RetentionTime = 6.764283333\n Charge = 2\n LabeledMass = 421.2483362\n PrecursorMass = 832.5058404 }) Item1 index value 0 { Apex = { Index = 17\n XVal = 1.850733333\n YVal = 12.9715029 }\n LeftLiftOff = Some { Index = 11\n XVal = 1.820683333\n YVal = 0.0 }\n LeftEnd = { Index = 15\n XVal = 1.8407\n YVal = 2.594316713 }\n RightL... Apex { Index = 17\n XVal = 1.850733333\n YVal = 12.9715029 } Index 17 XVal 1.850733333333 YVal 12.97150290299669 LeftLiftOff Some({ Index = 11\n XVal = 1.820683333\n YVal = 0.0 }) Value { Index = 11\n XVal = 1.820683333\n YVal = 0.0 } Index 11 XVal 1.820683333333 YVal 0 LeftEnd { Index = 15\n XVal = 1.8407\n YVal = 2.594316713 } Index 15 XVal 1.8407 YVal 2.5943167128161804 RightLiftOff Some({ Index = 22\n XVal = 1.8757\n YVal = 6.629763184 }) Value { Index = 22\n XVal = 1.8757\n YVal = 6.629763184 } Index 22 XVal 1.8757 YVal 6.629763184457715 RightEnd { Index = 22\n XVal = 1.8757\n YVal = 6.629763184 } Index 22 XVal 1.8757 YVal 6.629763184457715 LeftSidedConvolved True RightSidedConvolved True XData [ 1.8407, 1.845683333333, 1.850733333333, 1.855716666667, 1.8607, 1.86575, 1.870733333333, 1.8757 ] YData [ 2.5943167128161804, 33.725445996441294, 12.97150290299669, 23.20473619681991, 0, 3.315020657410628, 24.21335977039689, 6.629763184457715 ] 1 { Apex = { Index = 27\n XVal = 1.90075\n YVal = 7.782460951 }\n LeftLiftOff = Some { Index = 22\n XVal = 1.8757\n YVal = 6.629763184 }\n LeftEnd = { Index = 22\n XVal = 1.8757\n YVal = 6.629763184 }\n RightL... Apex { Index = 27\n XVal = 1.90075\n YVal = 7.782460951 } Index 27 XVal 1.90075 YVal 7.7824609511594645 LeftLiftOff Some({ Index = 22\n XVal = 1.8757\n YVal = 6.629763184 }) Value { Index = 22\n XVal = 1.8757\n YVal = 6.629763184 } Index 22 XVal 1.8757 YVal 6.629763184457715 LeftEnd { Index = 22\n XVal = 1.8757\n YVal = 6.629763184 } Index 22 XVal 1.8757 YVal 6.629763184457715 RightLiftOff Some({ Index = 34\n XVal = 1.93575\n YVal = 0.0 }) Value { Index = 34\n XVal = 1.93575\n YVal = 0.0 } Index 34 XVal 1.93575 YVal 0 RightEnd { Index = 32\n XVal = 1.9258\n YVal = 0.0 } Index 32 XVal 1.9258 YVal 0 LeftSidedConvolved True RightSidedConvolved True XData [ 1.8757, 1.880766666667, 1.885733333333, 1.890716666667, 1.895783333333, 1.90075, 1.905733333333, 1.910783333333, 1.915766666667, 1.920716666667, 1.9258 ] YData [ 6.629763184457715, 0, 19.601932004241007, 5.188475944794391, 23.347942041323222, 7.7824609511594645, 2.594174615768452, 18.159470837491654, 12.97118373783519, 3.314915145699615, 0 ] 2 { Apex = { Index = 40\n XVal = 1.965783333\n YVal = 7.782696222 }\n LeftLiftOff = Some { Index = 34\n XVal = 1.93575\n YVal = 0.0 }\n LeftEnd = { Index = 35\n XVal = 1.940816667\n YVal = 0.0 }\n RightLiftOff... Apex { Index = 40\n XVal = 1.965783333\n YVal = 7.782696222 } Index 40 XVal 1.965783333333 YVal 7.782696222085406 LeftLiftOff Some({ Index = 34\n XVal = 1.93575\n YVal = 0.0 }) Value { Index = 34\n XVal = 1.93575\n YVal = 0.0 } Index 34 XVal 1.93575 YVal 0 LeftEnd { Index = 35\n XVal = 1.940816667\n YVal = 0.0 } Index 35 XVal 1.940816666667 YVal 0 RightLiftOff Some({ Index = 45\n XVal = 1.99085\n YVal = 2.594240363 }) Value { Index = 45\n XVal = 1.99085\n YVal = 2.594240363 } Index 45 XVal 1.99085 YVal 2.594240363418976 RightEnd { Index = 43\n XVal = 1.9808\n YVal = 16.57454026 } Index 43 XVal 1.9808 YVal 16.574540264394955 LeftSidedConvolved True RightSidedConvolved True XData [ 1.940816666667, 1.9458, 1.950766666667, 1.955833333333, 1.9608, 1.965783333333, 1.970833333333, 1.975816666667, 1.9808 ] YData [ 0, 0, 7.7829121728370865, 12.539411434315184, 5.188368463205961, 7.782696222085406, 0, 0, 16.574540264394955 ] 3 { Apex = { Index = 49\n XVal = 2.01085\n YVal = 17.29544688 }\n LeftLiftOff = Some { Index = 45\n XVal = 1.99085\n YVal = 2.594240363 }\n LeftEnd = { Index = 47\n XVal = 2.0009\n YVal = 7.783070518 }\n Right... Apex { Index = 49\n XVal = 2.01085\n YVal = 17.29544

The peak model includes numerus information. Therefore we can mark the apices of the peaks we identified.


In [ ]:
// Code-Block 4
// Extract only apex coordinates (RT, intensity) from each detected subpeak
// while preserving associated metadata from `peaks`
let apices =
    peaks
    |> Array.map (fun peak ->
        fst peak
        |> Array.map (fun subpeak -> subpeak.Apex.XVal,subpeak.Apex.YVal), snd peak
    )

// For each trace: overlay detected apex points on top of baseline-corrected XIC
let apicesChart=
    Array.map2 (fun apice baseLineC ->
        [    
            Chart.Point(apice, Name = "apices")
            |> Chart.withMarkerStyle(Size = 15)
            Chart.Point(baseLineC, Name = "XIC")

        ]
        |> Chart.combine
        |> Chart.withXAxisStyle "Retention Time"
        |> Chart.withYAxisStyle "Intensity"
        |> Chart.withSize (900., 900.)
    ) (apices |> Array.map (fun (x,y) -> x)) baselineCorrection
apicesChart


We can then go ahead and characterize our peaks and quantify the area under the fitted curve.


In [44]:
// Code-Block 5
// Quantify one peak per trace using the RT of the most intense apex
let quantifiedXIC = 
    Array.map2 (fun rt (peak, metaData) -> 
        // Pick the detected peak at/near the target RT
        BioFSharp.Mz.Quantification.HULQ.getPeakBy peak rt
        // quantify peak of interest
        |> BioFSharp.Mz.Quantification.HULQ.quantifyPeak,
        metaData
    ) (apices |> Array.map (fun a -> a |> fst |> Array.maxBy snd |> fst)) peaks

// Extract only the quantified peak area values
let area = 
    quantifiedXIC
    |> Array.map (fun (x, metaData) ->  x.Area)
area


[ 1588.757120849495, 86.60081824026975, 38933.55580007476, 137.0931965191751, 4182.010301533513, 8698.082280223458, 18133.748441560998, 44276.3697061127, 411.31050606569914, 42492.61205773531, 29116.55081593598, 32396.941858300157, 40628.79681342983, 6835.18616826824, 31993.942555986454, 4189.014071334692, 22070.765805971874, 12612.32592585337, 664.9912930416041, 29116.560359433 ... (79 more) ]

The peak model gives us all the information we need for our peptide of interest. If we want to see what we quantified , we can take an 
exponential modified gaussian function (lol) using the parameters given by the peak model and plot it together with the previously extracted XIC.


In [ ]:
open BioFSharp.Mz
// Code-Block 6
// For each quantified peak, build an evaluable fitted-curve function (if a model exists)
// and keep the corresponding metadata attached.
let evaluation =
    quantifiedXIC
    |> Array.map (fun (peak, metaData) ->
        match peak.Model with
        // EMG fit: create function value evaluator from estimated EMG parameters
        | Some (Quantification.HULQ.PeakModel.EMG model) ->
            (Fitting.NonLinearRegression.Table.emgModel.GetFunctionValue (vector peak.EstimatedParams), metaData)
            |> Some
        // Gaussian fit: create function value evaluator from estimated Gaussian parameters
        | Some (Quantification.HULQ.PeakModel.Gaussian model) ->
            (Fitting.NonLinearRegression.Table.gaussModel.GetFunctionValue (vector peak.EstimatedParams), metaData)
            |> Some
        | None -> None
    )
        

In [ ]:
// Code-Block 7
// Evaluate fitted model values on baseline-corrected RT points, keeping metadata
let quantifiedArea =
    Array.map2 (fun p (e: ((float -> float)*ConsensLookup) option) ->
        if e.IsSome then
            (
            // For each (rt, intensity) point in baseline-corrected trace:
            // keep rt and replace intensity with model-predicted intensity
                p
                |> Array.map (fun (x, _) ->
                    x, fst(e.Value) x
                ),
                snd(e.Value)
            )
            |> Some
        else None
    ) baselineCorrection evaluation


let quantifiedAreaChart =
    Array.map2 (fun baselineC (qA: option<array<float * float>* ConsensLookup>) ->
        if qA.IsSome then
            let data,metadata = qA.Value
            [
                Chart.Point(baselineC, Name="XIC" )
                Chart.SplineArea(data, Name = "quantified XIC")
            ]
            |> Chart.combine
            |> Chart.withTitle($"Peptide:{metadata.PeptideSequence} Charge:{metadata.Charge} Retentiontime:{metadata.RetentionTime}")
            |> Chart.withXAxisStyle (TitleText = "Retention Time")
            |> Chart.withYAxisStyle "Intensity"
            |> Chart.withSize (900., 900.)
            |> Some
        else None
    ) baselineCorrection quantifiedArea
    |> Array.choose id

quantifiedAreaChart

### Wonderful! 
The result is speaking for itself :) The final step is to get you another .tsv file were all important data is stored.

In [54]:
let output = 
    consensLookup
    |> FSharpAux.IO.SeqIO.Seq.CSV "\t" true false
    |> fun csv -> System.IO.File.WriteAllLines("/home/paulinehans/Dokumente/relativeQuantification.tsv", csv)
output

## Questions
1. Why is a retention-time index from MS1 needed when extracting XICs from MS2-related queries?
2. What is the purpose of grouping by peptide sequence and charge before building consensLookup?
3. How might your quantification change if you choose a different rule than “highest matching score” within each group?
4. Why can baseline correction improve peak detection and downstream quantification?
5. Which signal characteristics does second-derivative peak detection emphasize, and when can it fail?
6. How does selecting the apex with highest intensity influence which peak is quantified?
7. How would you evaluate whether the fitted curve is biologically meaningful and not just mathematically good?



